# Building LLM

## Goal

This notebook is a hands-on journey to build a language model from scratch.

Each version introduces one new concept, allowing the model to evolve step by step while practicing language-model development.

---

## Version 8

In this version, we replace the gated recurrent processing from Version 7 with scaled dot-product attention.

The model keeps the same character vocabulary, fixed four-character context and trainable character embeddings used in the previous versions.

Instead of processing the context sequentially through a recurrent hidden state, the model now allows the character positions to interact directly through attention.

Because recurrence previously provided an implicit notion of order, Version 8 introduces trainable positional embeddings so that the model can distinguish where each character appears inside the context.

For every context position, the model creates:

- a **query**, representing what information that position is looking for;
- a **key**, representing the information available at that position for comparison;
- a **value**, representing the information that can be combined with information from other positions.

Queries are compared with keys to produce attention scores.

The scores are scaled and converted into attention weights with softmax.

These weights determine how the value vectors from the different context positions are combined.

The resulting representation of the final context position is used to predict the next character.

This version introduces scaled dot-product attention while preserving the same training data, fixed context and overall character-level language-modeling task.

## 1. Imports and Configuration

The Python standard library configures the execution environment before TensorFlow is imported.

GPU execution is disabled because this small model runs efficiently on the CPU and does not require CUDA. Low-level TensorFlow logs are suppressed to keep the notebook output clean.

TensorFlow provides tensor operations, trainable variables and automatic differentiation.

NumPy remains useful for reproducible data shuffling and sampling, while TensorFlow performs the model calculations and training.

The configuration collects the values that control the experiment.

`CONTEXT_LENGTH` defines how many previous characters are used to predict the next character.

`EMBEDDING_DIM` defines the size of the trainable vector used to represent each character.

`ATTENTION_DIM` defines the size of the query, key and value vectors used by the attention mechanism.

An explicit seed makes weight initialization, data splitting, mini-batch shuffling and text generation reproducible.

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import tensorflow as tf

tf.config.set_visible_devices([], "GPU")

In [2]:
SEED = 42
TRAIN_FRACTION = 0.8
BATCH_SIZE = 32
LEARNING_RATE = 1.0
EPOCHS = 100

CONTEXT_LENGTH = 4
EMBEDDING_DIM = 8
ATTENTION_DIM = 16

tf.keras.utils.set_random_seed(SEED)

## 2. Training Data

### Training text

The English corpus is included directly in the notebook. It provides the text from which the model learns character patterns.

The corpus is kept unchanged from Version 7 so that the effect of replacing gated recurrent processing with attention can be observed without changing the training data.

In [3]:
corpus = 'language models learn patterns from text.\na small model predicts what character may come next.\nwe begin with counting because counting is easy to inspect.\nthe model sees letters, spaces, and punctuation.\neach prediction comes from examples found in the training text.\nsimple systems help us understand more advanced systems.\nlater versions will learn parameters with neural networks.\nclear experiments make machine learning easier to study.'

print(corpus)
print("Characters:", len(corpus))

language models learn patterns from text.
a small model predicts what character may come next.
we begin with counting because counting is easy to inspect.
the model sees letters, spaces, and punctuation.
each prediction comes from examples found in the training text.
simple systems help us understand more advanced systems.
later versions will learn parameters with neural networks.
clear experiments make machine learning easier to study.
Characters: 440


### Vocabulary

The vocabulary is the set of symbols the model can represent. Because this is a character model, every letter, space, punctuation mark and newline is a token.

Each character is assigned an integer identifier.

As in Version 7, these identifiers are used to look up trainable embedding vectors.

In [4]:
vocabulary = sorted(set(corpus))
vocabulary_size = len(vocabulary)

character_to_id = {character: index for index, character in enumerate(vocabulary)}
id_to_character = {index: character for character, index in character_to_id.items()}

print("Vocabulary size:", vocabulary_size)
print("Vocabulary:", repr("".join(vocabulary)))
print("First mappings:", list(character_to_id.items())[:10])

Vocabulary size: 27
Vocabulary: '\n ,.abcdefghiklmnoprstuvwxy'
First mappings: [('\n', 0), (' ', 1), (',', 2), ('.', 3), ('a', 4), ('b', 5), ('c', 6), ('d', 7), ('e', 8), ('f', 9)]


### Context windows and numerical encoding

Each character is converted into its integer identifier.

For the text `modeling`:

`modeling`  
↓  
`[15, 17, 7, 8, 14, 12, 16, 10]`

The model keeps the fixed four-character context used in Version 7.

With a context length of 4, four adjacent identifiers form each input context:

- `mode -> l` becomes `[15, 17, 7, 8] -> 14`
- `odel -> i` becomes `[17, 7, 8, 14] -> 12`
- `deli -> n` becomes `[7, 8, 14, 12] -> 16`
- `elin -> g` becomes `[8, 14, 12, 16] -> 10`

The four identifiers in each context are the input. The following identifier is the target to predict.

As in Version 7, each identifier is first mapped to a trainable character embedding.

Unlike the recurrent model, the embeddings are no longer processed one at a time through a hidden state.

Instead, positional information is added to the embeddings and the resulting representations interact directly through attention.

The order of the four context positions is therefore represented explicitly rather than being provided implicitly by recurrent processing.

During generation, predicted identifiers are converted back into characters:

`[15, 17, 7, 8, 14, 12, 16, 10]`  
↓  
`modeling`

In [5]:
examples = [
    (corpus[index:index + CONTEXT_LENGTH], corpus[index + CONTEXT_LENGTH])
    for index in range(len(corpus) - CONTEXT_LENGTH)
]

input_ids = np.array([
    [character_to_id[character] for character in context]
    for context, _ in examples
], dtype=np.int32)

target_ids = np.array([character_to_id[target] for _, target in examples], dtype=np.int32)

print("Number of examples:", len(examples))
print("Input shape:", input_ids.shape)
print("Target shape:", target_ids.shape)
print("First 8 examples:", examples[:8])
print("First 8 input IDs:")
print(input_ids[:8])
print("First 8 target IDs:", target_ids[:8])

Number of examples: 436
Input shape: (436, 4)
Target shape: (436,)
First 8 examples: [('lang', 'u'), ('angu', 'a'), ('ngua', 'g'), ('guag', 'e'), ('uage', ' '), ('age ', 'm'), ('ge m', 'o'), ('e mo', 'd')]
First 8 input IDs:
[[14  4 16 10]
 [ 4 16 10 22]
 [16 10 22  4]
 [10 22  4 10]
 [22  4 10  8]
 [ 4 10  8  1]
 [10  8  1 15]
 [ 8  1 15 17]]
First 8 target IDs: [22  4 10  8  1 15 17  7]


## 3. Neural Model

### Trainable embeddings and attention projections

Version 8 replaces the gated recurrent computation from Version 7 with scaled dot-product attention.

Each character identifier still selects a trainable embedding vector.

For a context of four characters:

`[15, 17, 7, 8]`  
↓  
`4 character embedding vectors`

Unlike the GRU, attention does not process these vectors one at a time through a recurrent hidden state.

The model therefore adds a trainable positional embedding to each character embedding so that the four positions remain distinguishable.

The resulting representations are transformed into three different vectors for every position:

- a **query**, representing what information that position is looking for;
- a **key**, representing what information that position offers for comparison;
- a **value**, representing the information that can be combined with information from other positions.

Queries are compared with keys using dot products.

The resulting attention scores are divided by the square root of the attention dimension. This scaling keeps the values from becoming unnecessarily large as the vector dimension increases.

Softmax converts the scaled scores into attention weights.

Each row of attention weights sums to 1 and determines how strongly one position uses the value vectors from all positions in the context.

The weighted value vectors produce a contextual representation for every position.

For next-character prediction, the contextual representation of the final position is multiplied by an output weight matrix to produce one logit for every possible next character.

No bias terms, causal mask, residual connections or normalization are introduced yet, keeping the attention mechanism explicit and easy to inspect.

In [6]:
inputs = tf.convert_to_tensor(input_ids, dtype=tf.int32)

model_random = tf.random.Generator.from_seed(SEED)

embedding_matrix = tf.Variable(
    model_random.normal(
        shape=(vocabulary_size, EMBEDDING_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

position_embedding_matrix = tf.Variable(
    model_random.normal(
        shape=(CONTEXT_LENGTH, EMBEDDING_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

query_weights = tf.Variable(
    model_random.normal(
        shape=(EMBEDDING_DIM, ATTENTION_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

key_weights = tf.Variable(
    model_random.normal(
        shape=(EMBEDDING_DIM, ATTENTION_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

value_weights = tf.Variable(
    model_random.normal(
        shape=(EMBEDDING_DIM, ATTENTION_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

output_weights = tf.Variable(
    model_random.normal(
        shape=(ATTENTION_DIM, vocabulary_size),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)


def attention_forward(
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    output_weights,
    inputs
):
    embeddings = tf.gather(embedding_matrix, inputs)

    positioned_embeddings = (embeddings + position_embedding_matrix[tf.newaxis, :, :])

    queries = tf.matmul(positioned_embeddings, query_weights)
    keys = tf.matmul(positioned_embeddings, key_weights)
    values = tf.matmul(positioned_embeddings, value_weights)

    attention_scores = tf.matmul(queries, keys, transpose_b=True)

    scale = tf.sqrt(tf.cast(ATTENTION_DIM, tf.float32))
    scaled_attention_scores = attention_scores / scale

    attention_weights = tf.nn.softmax(scaled_attention_scores, axis=-1)

    attention_output = tf.matmul(attention_weights, values)

    final_representation = attention_output[:, -1, :]

    logits = tf.matmul(final_representation, output_weights)

    return attention_output, attention_weights, logits

In [7]:
example_embeddings = tf.gather(embedding_matrix, inputs[:1])

example_positioned_embeddings = ( example_embeddings + position_embedding_matrix[tf.newaxis, :, :])

example_queries = tf.matmul(example_positioned_embeddings, query_weights)

example_keys = tf.matmul(example_positioned_embeddings, key_weights)

example_values = tf.matmul(example_positioned_embeddings, value_weights)

example_attention_output, example_attention_weights, example_logits = attention_forward(
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    output_weights,
    inputs[:1]
)

trainable_parameters = (
    tf.size(embedding_matrix)
    + tf.size(position_embedding_matrix)
    + tf.size(query_weights)
    + tf.size(key_weights)
    + tf.size(value_weights)
    + tf.size(output_weights)
)

print("Input tensor shape:", inputs.shape)
print("Embedding matrix shape:", embedding_matrix.shape)
print("Embedded context shape:", example_embeddings.shape)
print("Position embedding matrix shape:", position_embedding_matrix.shape)
print("Positioned context shape:", example_positioned_embeddings.shape)

print("Query weight matrix shape:", query_weights.shape)
print("Key weight matrix shape:", key_weights.shape)
print("Value weight matrix shape:", value_weights.shape)

print("Queries shape:", example_queries.shape)
print("Keys shape:", example_keys.shape)
print("Values shape:", example_values.shape)

print("Attention weights shape:", example_attention_weights.shape)
print("Attention output shape:", example_attention_output.shape)

print("Output weight matrix shape:", output_weights.shape)
print("Logits shape:", example_logits.shape)
print("Trainable parameters:", trainable_parameters.numpy())

Input tensor shape: (436, 4)
Embedding matrix shape: (27, 8)
Embedded context shape: (1, 4, 8)
Position embedding matrix shape: (4, 8)
Positioned context shape: (1, 4, 8)
Query weight matrix shape: (8, 16)
Key weight matrix shape: (8, 16)
Value weight matrix shape: (8, 16)
Queries shape: (1, 4, 16)
Keys shape: (1, 4, 16)
Values shape: (1, 4, 16)
Attention weights shape: (1, 4, 4)
Attention output shape: (1, 4, 16)
Output weight matrix shape: (16, 27)
Logits shape: (1, 27)
Trainable parameters: 1064


### Training and validation split

The examples are divided into two separate groups:

- the training set is used to update the model parameters;
- the validation set is used to measure the loss on examples that do not update the parameters.

Each input example contains an ordered sequence of four character identifiers.

The same train/validation split used in Version 7 is preserved so that the new attention architecture can be compared without changing the data separation procedure.

Inside the model, the character identifiers are converted into embeddings and combined with trainable positional embeddings before attention is applied.

The indices are shuffled with a local random generator, making the split reproducible.

In [8]:
split_random = np.random.default_rng(SEED)

indices = split_random.permutation(len(inputs))
split_position = int(len(indices) * TRAIN_FRACTION)

train_indices = indices[:split_position]
validation_indices = indices[split_position:]

train_inputs = tf.gather(inputs, train_indices)
train_targets = tf.gather(target_ids, train_indices)

validation_inputs = tf.gather(inputs, validation_indices)
validation_targets = tf.gather(target_ids, validation_indices)

print("Training examples:", len(train_inputs))
print("Validation examples:", len(validation_inputs))
print("Training input shape:", train_inputs.shape)
print("Validation input shape:", validation_inputs.shape)

Training examples: 348
Validation examples: 88
Training input shape: (348, 4)
Validation input shape: (88, 4)


### Softmax probabilities

TensorFlow provides `tf.nn.softmax` to convert logits into probabilities.

- every probability is between 0 and 1;
- the probabilities for one input sum to 1;
- higher logits produce higher probabilities.

TensorFlow handles the numerical stability of this operation internally.

In [9]:
def softmax(logits):
    return tf.nn.softmax(logits, axis=1)

### Cross-entropy loss

Cross-entropy measures how much probability the model assigns to the correct next character.

Before calculating the loss, the context identifiers are mapped to their character embeddings.

Trainable positional embeddings are added so that the model can distinguish the four positions in the context.

The positioned embeddings are transformed into queries, keys and values.

The queries and keys produce scaled attention scores, and softmax converts these scores into attention weights.

The attention weights combine the value vectors to produce a contextual representation for every position.

The contextual representation of the final position is multiplied by the output weights to produce the logits.

TensorFlow calculates the cross-entropy directly from the logits and integer target IDs using a numerically stable operation.

A high probability for the correct character produces a low loss.

Training will update the character embeddings, positional embeddings, query weights, key weights, value weights and output weights to reduce this value.

In [10]:
def calculate_loss(
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    output_weights,
    inputs,
    targets
):
    _, _, logits = attention_forward(
        embedding_matrix,
        position_embedding_matrix,
        query_weights,
        key_weights,
        value_weights,
        output_weights,
        inputs
    )

    example_losses = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=targets, logits=logits)

    return tf.reduce_mean(example_losses)

In [11]:
initial_train_loss = calculate_loss(
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    output_weights,
    train_inputs,
    train_targets
)

initial_validation_loss = calculate_loss(
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    output_weights,
    validation_inputs,
    validation_targets
)

print("Initial train loss:", initial_train_loss.numpy())
print("Initial validation loss:", initial_validation_loss.numpy())

Initial train loss: 3.2958353
Initial validation loss: 3.2958367


### Training with mini-batch gradient descent

Training remains organized into epochs.

At each recorded epoch, training and validation loss are measured using the current model parameters.

Except at the final recorded epoch, the training examples are then shuffled and divided into mini-batches.

For every mini-batch:

1. `tf.GradientTape` records the attention forward calculations;
2. TensorFlow calculates the gradients automatically;
3. the character embeddings, positional embeddings, query, key and value projection matrices, and output weights are updated manually with gradient descent.

The validation examples are never used to update the model parameters.

The complete attention computation is recorded by `tf.GradientTape`.

Gradients therefore flow backward through the output projection, weighted value combinations, attention weights, query-key comparisons, query, key and value projections, positional embeddings and character embeddings.

The same query, key and value projection matrices are reused for every position in the context.

Version 8 keeps the same mini-batch training procedure while replacing gated recurrent computation with scaled dot-product attention.

### Mini-batches

A mini-batch is a small group of training examples.

The model updates its parameters after every mini-batch instead of processing all training examples together.

The final mini-batch may contain fewer examples than the configured batch size.

In [12]:
def create_batches(inputs, targets, batch_size):
    for start in range(0, len(inputs), batch_size):
        end = start + batch_size

        batch_inputs = inputs[start:end]
        batch_targets = targets[start:end]

        yield batch_inputs, batch_targets

In [13]:
def train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    initial_embedding_matrix,
    initial_position_embedding_matrix,
    initial_query_weights,
    initial_key_weights,
    initial_value_weights,
    initial_output_weights,
    learning_rate=1.0,
    batch_size=32,
    epochs=100,
    seed=42,
    print_every=10
):
    trained_embedding_matrix = tf.Variable(initial_embedding_matrix)

    trained_position_embedding_matrix = tf.Variable(initial_position_embedding_matrix)

    trained_query_weights = tf.Variable(initial_query_weights)

    trained_key_weights = tf.Variable(initial_key_weights)

    trained_value_weights = tf.Variable(initial_value_weights)

    trained_output_weights = tf.Variable(initial_output_weights)

    training_random = np.random.default_rng(seed)

    train_loss_history = []
    validation_loss_history = []

    for epoch in range(epochs + 1):
        train_loss = float(
            calculate_loss(
                trained_embedding_matrix,
                trained_position_embedding_matrix,
                trained_query_weights,
                trained_key_weights,
                trained_value_weights,
                trained_output_weights,
                train_inputs,
                train_targets
            ).numpy()
        )

        validation_loss = float(
            calculate_loss(
                trained_embedding_matrix,
                trained_position_embedding_matrix,
                trained_query_weights,
                trained_key_weights,
                trained_value_weights,
                trained_output_weights,
                validation_inputs,
                validation_targets
            ).numpy()
        )

        train_loss_history.append(train_loss)
        validation_loss_history.append(validation_loss)

        if print_every is not None and epoch % print_every == 0:
            print(
                f"Epoch {epoch:3d} | "
                f"Train loss: {train_loss:.4f} | "
                f"Validation loss: {validation_loss:.4f}"
            )

        if epoch == epochs:
            break

        shuffled_indices = training_random.permutation(len(train_inputs))

        shuffled_inputs = tf.gather(train_inputs, shuffled_indices)

        shuffled_targets = tf.gather(train_targets, shuffled_indices)

        for batch_inputs, batch_targets in create_batches(shuffled_inputs, shuffled_targets, batch_size):
            with tf.GradientTape() as tape:
                batch_loss = calculate_loss(
                    trained_embedding_matrix,
                    trained_position_embedding_matrix,
                    trained_query_weights,
                    trained_key_weights,
                    trained_value_weights,
                    trained_output_weights,
                    batch_inputs,
                    batch_targets
                )

            trainable_variables = [
                trained_embedding_matrix,
                trained_position_embedding_matrix,
                trained_query_weights,
                trained_key_weights,
                trained_value_weights,
                trained_output_weights
            ]

            gradients = tape.gradient(batch_loss, trainable_variables)

            gradients[0] = tf.convert_to_tensor(gradients[0])

            for variable, gradient in zip(trainable_variables, gradients):
                variable.assign_sub(learning_rate * gradient)

    return (
        trained_embedding_matrix,
        trained_position_embedding_matrix,
        trained_query_weights,
        trained_key_weights,
        trained_value_weights,
        trained_output_weights,
        train_loss_history,
        validation_loss_history
    )

In [14]:
(
    trained_embedding_matrix,
    trained_position_embedding_matrix,
    trained_query_weights,
    trained_key_weights,
    trained_value_weights,
    trained_output_weights,
    train_loss_history,
    validation_loss_history
) = train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    output_weights,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED
)

final_train_loss = train_loss_history[-1]
final_validation_loss = validation_loss_history[-1]

best_validation_epoch = int(np.argmin(validation_loss_history))

best_validation_loss = validation_loss_history[best_validation_epoch]

print()
print("Initial train loss:", initial_train_loss.numpy())
print("Final train loss:", final_train_loss)
print("Initial validation loss:", initial_validation_loss.numpy())
print("Final validation loss:", final_validation_loss)
print("Best validation loss:", best_validation_loss)
print("Best validation epoch:", best_validation_epoch)

Epoch   0 | Train loss: 3.2958 | Validation loss: 3.2958
Epoch  10 | Train loss: 3.2958 | Validation loss: 3.2958
Epoch  20 | Train loss: 3.2958 | Validation loss: 3.2958
Epoch  30 | Train loss: 3.2377 | Validation loss: 3.2457
Epoch  40 | Train loss: 2.8854 | Validation loss: 2.9831
Epoch  50 | Train loss: 2.8681 | Validation loss: 2.9792
Epoch  60 | Train loss: 2.9327 | Validation loss: 3.0737
Epoch  70 | Train loss: 2.8149 | Validation loss: 2.9190
Epoch  80 | Train loss: 2.7827 | Validation loss: 2.9149
Epoch  90 | Train loss: 2.7576 | Validation loss: 2.9223
Epoch 100 | Train loss: 2.7354 | Validation loss: 2.9271

Initial train loss: 3.2958353
Final train loss: 2.7353932857513428
Initial validation loss: 3.2958367
Final validation loss: 2.927145481109619
Best validation loss: 2.884859323501587
Best validation epoch: 74


### Reading the losses

At initialization, both training and validation loss are approximately `3.2958`.

The model initially changes very little, but after the first part of training the loss begins to decrease as the attention parameters learn useful relationships between the context positions.

The training loss decreases from approximately `3.2958` to `2.7354`.

The validation loss reaches its lowest value of approximately `2.8849` at epoch `74`.

After this point, the training loss continues to decrease while the validation loss begins to increase slightly.

This indicates that the model continues fitting the training examples without producing the same improvement on the validation examples.

The final validation loss is approximately `2.9271`, which is higher than the best validation loss reached during training.

This small gap between the best and final validation losses illustrates why validation performance should be monitored independently from training performance.

The result also shows that scaled dot-product attention can learn useful character relationships even in this very small model and with the same fixed four-character context used by the recurrent versions.

### Learned probabilities

After training, the attention model produces a probability distribution for the next character from an ordered four-character context.

The character identifiers are mapped to trainable embeddings and combined with positional embeddings.

The positioned representations are transformed into queries, keys and values.

Scaled dot-product attention allows every context position to combine information from the other positions.

The contextual representation of the final position is converted into logits and then into probabilities with softmax.

The example below inspects the learned next-character distribution for the context `mode`.

In [15]:
example_context = "mode"

example_context_ids = tf.constant(
    [[character_to_id[character] for character in example_context]],
    dtype=tf.int32
)

(
    example_attention_output,
    example_attention_weights,
    example_logits
) = attention_forward(
    trained_embedding_matrix,
    trained_position_embedding_matrix,
    trained_query_weights,
    trained_key_weights,
    trained_value_weights,
    trained_output_weights,
    example_context_ids
)

learned_probabilities = softmax(example_logits)[0].numpy()

sorted_probabilities = sorted(zip(vocabulary, learned_probabilities), key=lambda item: item[1], reverse=True)

print("Context:", repr(example_context))
print("Attention output shape:", example_attention_output.shape)
print("Attention weights shape:", example_attention_weights.shape)
print()

for character, probability in sorted_probabilities:
    print(repr(character), round(float(probability), 4))

print()
print("Total probability:", learned_probabilities.sum())

Context: 'mode'
Attention output shape: (1, 4, 16)
Attention weights shape: (1, 4, 4)

' ' 0.0897
'e' 0.0879
'a' 0.0698
'n' 0.061
's' 0.0588
't' 0.0581
'l' 0.0509
'o' 0.05
'i' 0.049
'r' 0.046
'p' 0.0414
'm' 0.0413
'u' 0.0397
'c' 0.033
'd' 0.0318
'.' 0.031
'h' 0.0252
'g' 0.0231
'w' 0.0208
'x' 0.0188
',' 0.0144
'f' 0.0137
'y' 0.0129
'b' 0.0106
'v' 0.01
'k' 0.0087
'\n' 0.0024

Total probability: 1.0


### Inspecting attention weights

Unlike the recurrent models, attention produces an explicit matrix describing how strongly every context position uses information from the other positions.

For a four-character context, the attention matrix has shape `4 x 4`.

Each row corresponds to one query position.

Each column corresponds to one key position.

The values in every row sum to 1 because softmax converts the scaled attention scores into a probability-like distribution.

The example below displays the attention weights learned for the context `mode`.

The final row is especially important for next-character prediction because the contextual representation of the final position is the one passed to the output layer.

In [16]:
print("Context positions:", list(example_context))
print()

print("Attention weights:")
print(np.round(example_attention_weights[0].numpy(), 4))

print()
print("Final-position attention:")
for character, weight in zip(example_context, example_attention_weights[0, -1].numpy()):
    print(repr(character), round(float(weight), 4))

print()
print("Final row sum:", example_attention_weights[0, -1].numpy().sum())

Context positions: ['m', 'o', 'd', 'e']

Attention weights:
[[0.2471 0.2603 0.1954 0.2973]
 [0.2464 0.2612 0.1891 0.3033]
 [0.2499 0.2522 0.2395 0.2583]
 [0.2422 0.2643 0.1634 0.3301]]

Final-position attention:
'm' 0.2422
'o' 0.2643
'd' 0.1634
'e' 0.3301

Final row sum: 0.99999994


## 4. Generator

The trained attention model can now generate new text one character at a time.

Generation remains autoregressive.

For every prediction:

1. the four most recent characters form the current context;
2. their identifiers are mapped to trainable character embeddings;
3. positional embeddings are added to preserve the order of the context;
4. the positioned representations are transformed into queries, keys and values;
5. scaled dot-product attention combines information across the four context positions;
6. the contextual representation of the final position produces the next-character logits;
7. softmax converts the logits into probabilities;
8. one character is sampled from the probability distribution;
9. the sampled character is appended to the generated text and the four-character context window moves forward.

The attention calculation is performed again for every new context window, matching the way the model was trained.

A local NumPy random generator keeps text generation reproducible without changing the global random state.

### Sample the next character

The current four-character context is converted into numerical identifiers.

The identifiers are mapped to character embeddings and combined with positional embeddings.

Scaled dot-product attention produces contextual representations for the four positions.

The representation of the final position is transformed into logits and probabilities.

A local random generator samples one identifier from this learned probability distribution and converts it back into a character.

In [17]:
def sample_next_character(
    context,
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    output_weights,
    random_generator
):
    context_ids = tf.constant(
        [[character_to_id[character] for character in context]],
        dtype=tf.int32
    )

    _, _, logits = attention_forward(
        embedding_matrix,
        position_embedding_matrix,
        query_weights,
        key_weights,
        value_weights,
        output_weights,
        context_ids
    )

    probabilities = softmax(logits)[0].numpy()

    next_id = random_generator.choice(
        vocabulary_size,
        p=probabilities
    )

    return id_to_character[next_id]

In [18]:
sample_random = np.random.default_rng(SEED)

example_context = "mode"

print("Context:", repr(example_context))

for _ in range(5):
    sampled_character = sample_next_character(
        example_context,
        trained_embedding_matrix,
        trained_position_embedding_matrix,
        trained_query_weights,
        trained_key_weights,
        trained_value_weights,
        trained_output_weights,
        sample_random
    )

    print("Sampled character:", repr(sampled_character))

Context: 'mode'
Sampled character: 'r'
Sampled character: 'i'
Sampled character: 't'
Sampled character: 'p'
Sampled character: ','


### Generate text

Text generation starts from a four-character context.

At every step, the attention model processes the current context and samples the next character.

The sampled character is appended to the output.

The oldest context character is then removed, causing the fixed context window to slide forward by one position.

For example:

`mode -> sampled character`

then:

`ode? -> next sampled character`

and so on.

For every new context window, character embeddings and positional embeddings are combined and the attention calculation is performed again.

The model therefore generates text autoregressively while using scaled dot-product attention inside every context window.

In [19]:
def generate_text(
    starting_context,
    number_of_characters,
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    output_weights,
    seed=42
):
    if len(starting_context) != CONTEXT_LENGTH:
        raise ValueError(
            f"starting_context must contain exactly "
            f"{CONTEXT_LENGTH} characters"
        )

    generated_text = starting_context
    generation_random = np.random.default_rng(seed)

    for _ in range(number_of_characters):
        current_context = generated_text[-CONTEXT_LENGTH:]

        next_character = sample_next_character(
            current_context,
            embedding_matrix,
            position_embedding_matrix,
            query_weights,
            key_weights,
            value_weights,
            output_weights,
            generation_random
        )

        generated_text += next_character

    return generated_text

In [20]:
generated_text = generate_text(
    starting_context="mode",
    number_of_characters=300,
    embedding_matrix=trained_embedding_matrix,
    position_embedding_matrix=trained_position_embedding_matrix,
    query_weights=trained_query_weights,
    key_weights=trained_key_weights,
    value_weights=trained_value_weights,
    output_weights=trained_output_weights,
    seed=SEED
)

print(generated_text)

modereto yor ievlneam tnrdwtsai apnwefia.
apesoesseeo.
 topoil  nilleeh  fahna deomonesa ,ni l oeeenf .
tpaxpnia,sh ei.
pngiml e l.
 nawnen xir keveiaeld stam ocopo ua mcsrawdl u  ixtrstl rie rawa.
.
msaertl  a.
 knm sop .
eeknwdu mi ahtlsrer .
ralesgeeeotiacrr  naudav e.
 
a nofelps acdme fl s u utxron


## 5. Tests

These assertions verify the context windows, TensorFlow tensors, trainable attention parameters, data split, model shapes, attention weights, probability distributions, gradients, training behavior and reproducibility.

The tests verify that:

- context windows and targets are constructed correctly;
- all six trainable parameter matrices are TensorFlow variables;
- character embeddings, positional embeddings, queries, keys, values, attention outputs and logits have the expected shapes;
- the model contains exactly 1064 trainable parameters;
- the attention matrix has shape `4 x 4` for each example;
- attention weights remain finite and between 0 and 1;
- every row of attention weights sums to 1;
- training and validation sets remain separate;
- training and validation loss histories contain the expected number of measurements;
- training reduces the loss and all recorded losses remain finite;
- next-character probability distributions sum to 1;
- TensorFlow produces finite gradients for every trainable parameter matrix;
- every trainable parameter matrix changes during training;
- repeating training from the same initial parameters with the same seed produces the same parameters and loss histories;
- autoregressive generation is reproducible with the same starting context and seed.

The tests also verify the requested generated-text length and starting context.

In [21]:
assert len(examples) == len(corpus) - CONTEXT_LENGTH
assert len(input_ids) == len(examples)
assert len(target_ids) == len(examples)

assert input_ids.shape == (len(examples), CONTEXT_LENGTH)
assert target_ids.shape == (len(examples),)

assert examples[0][0] == corpus[:CONTEXT_LENGTH]
assert examples[0][1] == corpus[CONTEXT_LENGTH]
assert examples[1][0] == corpus[1:1 + CONTEXT_LENGTH]
assert examples[1][1] == corpus[1 + CONTEXT_LENGTH]

assert tf.is_tensor(inputs)

# Trainable parameter checks

initial_parameter_variables = [
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    output_weights
]

trained_parameter_variables = [
    trained_embedding_matrix,
    trained_position_embedding_matrix,
    trained_query_weights,
    trained_key_weights,
    trained_value_weights,
    trained_output_weights
]

assert all(isinstance(variable, tf.Variable) for variable in initial_parameter_variables)
assert all(isinstance(variable, tf.Variable) for variable in trained_parameter_variables)

assert inputs.shape == (len(examples), CONTEXT_LENGTH)

assert embedding_matrix.shape == (vocabulary_size, EMBEDDING_DIM)
assert position_embedding_matrix.shape == (CONTEXT_LENGTH, EMBEDDING_DIM)

assert query_weights.shape == (EMBEDDING_DIM, ATTENTION_DIM)
assert key_weights.shape == (EMBEDDING_DIM, ATTENTION_DIM)
assert value_weights.shape == (EMBEDDING_DIM, ATTENTION_DIM)

assert output_weights.shape == (ATTENTION_DIM, vocabulary_size)

for initial_variable, trained_variable in zip(initial_parameter_variables, trained_parameter_variables):
    assert trained_variable.shape == initial_variable.shape

# Trainable parameter count

expected_trainable_parameters = (
    vocabulary_size * EMBEDDING_DIM
    + CONTEXT_LENGTH * EMBEDDING_DIM
    + 3 * EMBEDDING_DIM * ATTENTION_DIM
    + ATTENTION_DIM * vocabulary_size
)

assert expected_trainable_parameters == 1064
assert trainable_parameters.numpy() == expected_trainable_parameters

# Forward-pass shape checks

assert example_embeddings.shape == (1, CONTEXT_LENGTH, EMBEDDING_DIM)
assert example_positioned_embeddings.shape == (1, CONTEXT_LENGTH, EMBEDDING_DIM)

assert example_queries.shape == (1, CONTEXT_LENGTH, ATTENTION_DIM)
assert example_keys.shape == (1, CONTEXT_LENGTH, ATTENTION_DIM)
assert example_values.shape == (1, CONTEXT_LENGTH, ATTENTION_DIM)

assert example_attention_weights.shape == (1, CONTEXT_LENGTH, CONTEXT_LENGTH)
assert example_attention_output.shape == (1, CONTEXT_LENGTH, ATTENTION_DIM)
assert example_logits.shape == (1, vocabulary_size)

# Attention checks

attention_test_context = tf.constant(
    [[character_to_id[character] for character in "mode"]],
    dtype=tf.int32
)

attention_test_output, attention_test_weights, attention_test_logits = attention_forward(
    trained_embedding_matrix,
    trained_position_embedding_matrix,
    trained_query_weights,
    trained_key_weights,
    trained_value_weights,
    trained_output_weights,
    attention_test_context
)

assert attention_test_output.shape == (1, CONTEXT_LENGTH, ATTENTION_DIM)
assert attention_test_weights.shape == (1, CONTEXT_LENGTH, CONTEXT_LENGTH)
assert attention_test_logits.shape == (1, vocabulary_size)

assert np.all(np.isfinite(attention_test_output.numpy()))
assert np.all(np.isfinite(attention_test_weights.numpy()))

assert np.all(attention_test_weights.numpy() >= 0.0)
assert np.all(attention_test_weights.numpy() <= 1.0)

attention_row_sums = tf.reduce_sum(attention_test_weights, axis=-1).numpy()

assert np.allclose(attention_row_sums, np.ones((1, CONTEXT_LENGTH)), atol=1e-6)

# Training and validation checks

assert len(train_inputs) + len(validation_inputs) == len(inputs)
assert len(train_inputs) == len(train_targets)
assert len(validation_inputs) == len(validation_targets)

assert len(np.intersect1d(train_indices, validation_indices)) == 0

assert len(train_loss_history) == EPOCHS + 1
assert len(validation_loss_history) == EPOCHS + 1

assert final_train_loss < float(initial_train_loss.numpy())
assert final_validation_loss < float(initial_validation_loss.numpy())

assert np.all(np.isfinite(train_loss_history))
assert np.all(np.isfinite(validation_loss_history))

assert best_validation_epoch == int(np.argmin(validation_loss_history))
assert best_validation_loss == validation_loss_history[best_validation_epoch]

# Probability checks

probability_context = tf.constant(
    [[character_to_id[character] for character in "mode"]],
    dtype=tf.int32
)

_, _, probability_logits = attention_forward(
    trained_embedding_matrix,
    trained_position_embedding_matrix,
    trained_query_weights,
    trained_key_weights,
    trained_value_weights,
    trained_output_weights,
    probability_context
)

probability_values = softmax(probability_logits)[0].numpy()

assert abs(probability_values.sum() - 1.0) < 1e-6
assert np.all(np.isfinite(probability_values))
assert np.all(probability_values >= 0.0)
assert np.all(probability_values <= 1.0)

# Gradient checks

gradient_inputs = train_inputs[:BATCH_SIZE]
gradient_targets = train_targets[:BATCH_SIZE]

with tf.GradientTape() as tape:
    gradient_loss = calculate_loss(
        embedding_matrix,
        position_embedding_matrix,
        query_weights,
        key_weights,
        value_weights,
        output_weights,
        gradient_inputs,
        gradient_targets
    )

gradients = tape.gradient(gradient_loss, initial_parameter_variables)

assert all(gradient is not None for gradient in gradients)

dense_gradients = [tf.convert_to_tensor(gradient) for gradient in gradients]

for gradient, variable in zip(dense_gradients, initial_parameter_variables):
    assert gradient.shape == variable.shape
    assert np.all(np.isfinite(gradient.numpy()))

# Deterministic retraining

(
    repeated_embedding_matrix,
    repeated_position_embedding_matrix,
    repeated_query_weights,
    repeated_key_weights,
    repeated_value_weights,
    repeated_output_weights,
    repeated_train_history,
    repeated_validation_history
) = train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    output_weights,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED,
    print_every=None
)

repeated_parameter_variables = [
    repeated_embedding_matrix,
    repeated_position_embedding_matrix,
    repeated_query_weights,
    repeated_key_weights,
    repeated_value_weights,
    repeated_output_weights
]

for trained_variable, repeated_variable in zip(trained_parameter_variables, repeated_parameter_variables):
    assert np.allclose(trained_variable.numpy(), repeated_variable.numpy())

assert np.allclose(train_loss_history, repeated_train_history)
assert np.allclose(validation_loss_history, repeated_validation_history)

# Parameter-update checks

for initial_variable, trained_variable in zip(initial_parameter_variables, trained_parameter_variables):
    assert np.max(np.abs(initial_variable.numpy() - trained_variable.numpy())) > 0.0

# Generation reproducibility

first_generation = generate_text(
    starting_context="mode",
    number_of_characters=30,
    embedding_matrix=trained_embedding_matrix,
    position_embedding_matrix=trained_position_embedding_matrix,
    query_weights=trained_query_weights,
    key_weights=trained_key_weights,
    value_weights=trained_value_weights,
    output_weights=trained_output_weights,
    seed=10
)

second_generation = generate_text(
    starting_context="mode",
    number_of_characters=30,
    embedding_matrix=trained_embedding_matrix,
    position_embedding_matrix=trained_position_embedding_matrix,
    query_weights=trained_query_weights,
    key_weights=trained_key_weights,
    value_weights=trained_value_weights,
    output_weights=trained_output_weights,
    seed=10
)

assert first_generation == second_generation
assert len(first_generation) == CONTEXT_LENGTH + 30
assert first_generation.startswith("mode")

print("All checks passed.")

All checks passed.


## Notes

- The model remains a character-level neural language model.
- The training corpus and vocabulary remain unchanged from Version 7.
- Each prediction still uses a fixed context of four previous characters.
- Character identifiers are mapped to trainable 8-dimensional embedding vectors.
- Version 8 replaces gated recurrent processing with scaled dot-product attention.
- Trainable positional embeddings are added so that the model can distinguish the four positions in the context.
- The positioned embeddings are transformed into queries, keys and values.
- Queries are compared with keys using dot products.
- Attention scores are divided by the square root of the attention dimension before softmax is applied.
- Softmax converts the scaled scores into attention weights.
- Each row of the attention matrix sums to 1.
- Attention weights determine how value vectors from the different context positions are combined.
- The contextual representation of the final position is used to predict the next character.
- No bias terms are used.
- No causal mask is used yet.
- No residual connections, normalization or feed-forward network are used yet.
- The model learns six trainable parameter matrices:
  - the character embedding matrix;
  - the positional embedding matrix;
  - the query weight matrix;
  - the key weight matrix;
  - the value weight matrix;
  - the output weight matrix.
- The same query, key and value weight matrices are reused at every position in the context.
- Gradients for all trainable parameters are computed automatically with `tf.GradientTape`, and the parameters are updated manually with gradient descent.
- The model contains 1064 trainable parameters, compared with 1800 in Version 7.
- Training and validation examples remain reproducibly separated.
- Training examples are shuffled before each sequence of mini-batch updates.
- Mini-batches are used to update the model parameters.
- Training loss decreases from approximately `3.2958` to `2.7354`.
- Validation loss decreases from approximately `3.2958` to `2.9271`.
- The best validation loss is approximately `2.8849` at epoch 74.
- After epoch 74, training loss continues to decrease while validation loss increases slightly, suggesting mild overfitting.
- The final epoch-100 parameters are intentionally retained for generation.
- Generation remains autoregressive and uses a sliding four-character context window.
- Attention is recomputed for every new context window, matching the training procedure.
- Attention weights can be inspected directly to observe how the context positions combine information.
- The same seed makes parameter initialization, data splitting, mini-batch shuffling, training and generation reproducible within the same software environment.
- The tests verify attention parameter shapes, positional representations, attention weights, probability distributions, gradients, parameter updates and reproducibility.

Version 8 introduces scaled dot-product attention while preserving the same fixed four-character language-modeling setup used in the previous versions.

The model no longer carries information through a recurrent hidden state. Instead, every context position can directly compare its representation with the other positions and combine their value vectors through learned attention weights.

Removing recurrence also removes the implicit representation of sequence order, so Version 8 introduces trainable positional embeddings to preserve positional information explicitly.

The attention mechanism is still intentionally minimal. It uses a single attention computation and does not yet include causal masking, residual connections, normalization, feed-forward processing or multiple attention heads.

The fixed context, small corpus, absence of bias terms and lack of early stopping or checkpoint restoration continue to keep the model easy to inspect.

Version 8 therefore completes the transition from recurrent sequence processing to direct attention-based information exchange and prepares the construction of a Transformer block in the next version.

Future versions will introduce new components and gradually evolve the architecture.